## FEATURE ENGINEERING AND LABEL PREPARATION

In [ ]:
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler, Imputer
)
from pyspark.ml import Pipeline

df3 = df3.withColumn(
    "crime_category",
    F.when(F.col("Primary Type").isin(
        "BATTERY", "ASSAULT", "ROBBERY",
        "CRIM SEXUAL ASSAULT", "WEAPONS VIOLATION",
        "HOMICIDE", "KIDNAPPING"
    ), "VIOLENT")
    .when(F.col("Primary Type").isin(
        "NARCOTICS", "OTHER NARCOTIC VIOLATION"
    ), "DRUG")
    .otherwise("PROPERTY")   # THEFT, BURGLARY, FRAUD, OTHER etc -> PROPERTY
)

print("Crime category distribution (3 classes):")
df3.groupBy("crime_category").count().orderBy(F.desc("count")).show()
print("Number of classes:", df3.select("crime_category").distinct().count())

Crime category distribution (3 classes):
+--------------+-------+
|crime_category|  count|
+--------------+-------+
|      PROPERTY|3600260|
|       VIOLENT|1854009|
|          DRUG| 556189|
+--------------+-------+

Number of classes: 3


In [ ]:
# 2B - Rich feature engineering: amplify the signals the DT already found

df3 = df3.withColumn(
    # Night flag: Drug crimes peak 10pm-4am, Property crimes 8pm-2am
    # Violent crimes spike 9pm-midnight on weekends
    "is_night",
    F.when((F.col("hour") >= 22) | (F.col("hour") < 4), 1).otherwise(0)
).withColumn(
    # Late evening: 6pm-10pm - peak for street robbery and assault
    "is_evening",
    F.when((F.col("hour") >= 18) & (F.col("hour") < 22), 1).otherwise(0)
).withColumn(
    # Business hours: fraud and theft from stores peak 9am-5pm weekdays
    "is_business_hours",
    F.when(
        (F.col("hour") >= 9) & (F.col("hour") < 17) &
        (~F.col("day_of_week").isin([1, 7])), 1
    ).otherwise(0)
).withColumn(
    # Weekend flag
    "is_weekend",
    F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)
).withColumn(
    # Season: drug arrests spike in summer outdoors
    "season",
    F.when(F.col("month").isin([12, 1, 2]), 1)
     .when(F.col("month").isin([3, 4, 5]),  2)
     .when(F.col("month").isin([6, 7, 8]),  3)
     .otherwise(4)
).withColumn(
    # Lat-lon grid cell: snaps to 0.005-degree(~500m) grid
    # Drug hotspots occupy specific blocks
    "lat_grid",
    F.round(F.floor(F.col("Latitude")  / 0.005) * 0.005, 4)
).withColumn(
    "lon_grid",
    F.round(F.floor(F.col("Longitude") / 0.005) * 0.005, 4)
).withColumn(
    # Neighbourhood signature: unique float per ~500m grid cell
    "geo_cell",
    F.round(F.col("lat_grid") * F.col("lon_grid"), 6)
).withColumn(
    # Hour x lat interaction: drug hotspots active at different hours
    "hour_x_lat",
    F.round(F.col("hour") * F.col("Latitude"), 4)
).withColumn(
    # Hour x lon interaction
    "hour_x_lon",
    F.round(F.col("hour") * F.col("Longitude"), 4)
).withColumn(
    # District x hour: captures precinct-level time patterns
    "district_x_hour",
    F.col("District") * F.col("hour")
).withColumn(
    # Ward x night: which wards have late-night crime concentration
    "ward_x_night",
    F.col("Ward") * F.col("is_night")
)

print("Feature engineering complete.")
print("Total engineered features: 21")

Feature engineering complete.
Total engineered features: 21


In [ ]:
# Label encoding
if "label" in df3.columns:
    df3 = df3.drop("label")

label_indexer = StringIndexer(
    inputCol="crime_category",
    outputCol="label",
    handleInvalid="keep"
)
label_model = label_indexer.fit(df3)
df3         = label_model.transform(df3)

label_mapping = pd.DataFrame({
    "label_index":    range(len(label_model.labels)),
    "crime_category": label_model.labels
})
print("\nLabel mapping (3 classes):")
print(label_mapping)
label_mapping.to_csv(OUT_DIR + "/label_mapping.csv", index=False)


Label mapping (3 classes):
   label_index crime_category
0            0       PROPERTY
1            1        VIOLENT
2            2           DRUG


In [ ]:
# Full feature set: 21 features
num_cols = [
    # Core location and time
    "Latitude", "Longitude", "hour", "day_of_week", "month", "year_num",
    "Beat", "District", "Ward", "Community Area",
    # Engineered time features
    "is_night", "is_evening", "is_business_hours", "is_weekend", "season",
    # Engineered location features
    "lat_grid", "lon_grid", "geo_cell",
    # Interaction features
    "hour_x_lat", "hour_x_lon", "district_x_hour", "ward_x_night"
]

In [ ]:
# Impute, assemble, scale
imputer = Imputer(
    inputCols=num_cols,
    outputCols=[c + "_imp" for c in num_cols],
    strategy="median"
)
assembler = VectorAssembler(
    inputCols=[c + "_imp" for c in num_cols],
    outputCol="features_raw"
)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

In [ ]:
prep_pipeline = Pipeline(stages=[imputer, assembler, scaler])

# Persist df3 only for pipeline fitting - it will be released immediately after
df3 = df3.persist()
df3.count()
print("df3 persisted. Rows:", df3.count())

prep_model = prep_pipeline.fit(df3)

# Apply transformation and split into train/test IN THE SAME STEP

df_transformed = prep_model.transform(df3).select(
    "ID", "Year", "year_num", "Primary Type", "crime_category",
    "crime_group", "Arrest", "Latitude", "Longitude",
    "hour", "day_of_week", "month", "District", "Beat",
    "is_night", "is_evening", "is_business_hours", "is_weekend", "season",
    "label", "features"
)

df3 persisted. Rows: 6010458


In [ ]:
df_sample_train = df_transformed.sample(False, 0.20, seed=42).persist()
df_sample_train.count()

train_df, test_df = df_sample_train.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist()
train_df.count()
print("Train rows:", train_df.count())

test_df = test_df.persist()
test_df.count()
print("Test rows:", test_df.count())

df_sample_train.unpersist()

# df_feat kept as lazy reference for Section 9 exports
df_feat = df_transformed

print("\nClass distribution in training set:")
train_df.groupBy("crime_category", "label").count().orderBy("label").show()

Train rows: 963191
Test rows: 241315

Class distribution in training set:
+--------------+-----+------+
|crime_category|label| count|
+--------------+-----+------+
|      PROPERTY|  0.0|576857|
|       VIOLENT|  1.0|297246|
|          DRUG|  2.0| 89088|
+--------------+-----+------+

